In [1]:
# queries = [
#     "What is the capital of France?",
#     "How do I bake a chocolate cake?"
# ]
# passages = [
#     "The capital of France is Paris.",
#     "To bake a chocolate cake, you will need flour, cocoa powder, eggs, and butter."
# ]

# nv_embed = get_embedding_model("nvidia/NV-Embed-v2")
# query_embeddings, passage_embeddings = nv_embed(queries, passages)
# print(query_embeddings.shape, passage_embeddings.shape)


In [2]:
# queries = [
#     "What are some ways to reduce stress?",
#     "What are the benefits of drinking green tea?"
# ]
# docs = [
#     "Stress can be reduced through exercise, meditation, and relaxation techniques.",
#     "Green tea is rich in antioxidants and may promote heart health."
# ]

# stella = get_embedding_model("dunzhang/stella_en_1.5B_v5")
# query_embeddings, doc_embeddings = stella(queries, docs)
# print(query_embeddings.shape, doc_embeddings.shape)


In [2]:
from utils.utils import load_from_pickle

testset = load_from_pickle('validated_testset')

In [16]:
import pandas as pd

# Set Pandas display options to allow full column output
pd.set_option('display.max_colwidth', None)  # Disable truncation of column contents
pd.set_option('display.max_rows', None)  # Show all rows if needed
print(testset.iloc[28].to_string())

question                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          Here is a question that can be fully answered from the given context:\n\n"According to the Undang-Undang Nomor 23 Tahun 2007, what are the minimum facilities that are considered as 'fasilitas umum' in a station?"\n\nThis question can be answered by looking at th

In [13]:
def is_metadata_empty(row):
    metadata = row['metadata']
    if not metadata or metadata[0].get('id') == '' or metadata[0].get('page_number') == '':
        return True
    return False

# Filter rows with missing metadata
missing_metadata_rows = testset[testset.apply(is_metadata_empty, axis=1)]
print(f"Number of rows with missing metadata: {len(missing_metadata_rows)}")

Number of rows with missing metadata: 15


In [3]:
from utils.embed import split_documents, get_embedding_model, store_embeddings, perform_query
from utils.utils import load_from_pickle

In [4]:
# Load all documents
all_documents = load_from_pickle("all_documents")

# Define chunk sizes and overlaps for experimentation
chunk_sizes = [128, 256, 512, 1024]
chunk_overlaps = [0, 64, 128, 256, 512]

# Split documents
splits = split_documents(all_documents, chunk_sizes, chunk_overlaps)

# Initialize the embedding model
embed_model_bge = get_embedding_model(model="BAAI/bge-m3")

Splitting documents with chunk size 128 and overlap 0...
Split 46409 documents into 690735 chunks
Splitting documents with chunk size 128 and overlap 64...
Split 46409 documents into 955529 chunks
Splitting documents with chunk size 256 and overlap 0...
Split 46409 documents into 330288 chunks
Splitting documents with chunk size 256 and overlap 64...
Split 46409 documents into 372008 chunks
Splitting documents with chunk size 256 and overlap 128...
Split 46409 documents into 506835 chunks
Splitting documents with chunk size 512 and overlap 0...
Split 46409 documents into 189299 chunks
Splitting documents with chunk size 512 and overlap 64...
Split 46409 documents into 195860 chunks
Splitting documents with chunk size 512 and overlap 128...
Split 46409 documents into 211081 chunks
Splitting documents with chunk size 512 and overlap 256...
Split 46409 documents into 263903 chunks
Splitting documents with chunk size 1024 and overlap 0...
Split 46409 documents into 121005 chunks
Splitting 

In [5]:
# Store embeddings without HNSW
qdrant_manager_dense = store_embeddings(embed_model_bge, splits, index_name="dense-bge-m3", use_hnsw=False)

Configuring Qdrant with vector size=1024, distance=Cosine, HNSW=False...


AttributeError: 'QdrantIndexManager' object has no attribute 'check_index_exists'

In [ ]:
# Perform a query on the dense embeddings
perform_query(qdrant_manager_dense, query="Apa itu capital Jawa Barat?", k=5)

In [ ]:
# Store embeddings with HNSW
qdrant_manager_hnsw = store_embeddings(embed_model_bge, splits, index_name="hnsw-bge-m3", use_hnsw=True)

In [ ]:
# Perform a query on the HNSW embeddings
perform_query(qdrant_manager_hnsw, query="Apa itu capital Jawa Barat?", k=5)

In [5]:
from utils.database.qdrant import QdrantIndexManager
from utils.llm import get_embedding_model

embed_model = get_embedding_model(model="BAAI/bge-m3")
COLLECTION_NAME = "uu-all-512-token"
qdrant_index_manager = QdrantIndexManager(embed_model, index_name=COLLECTION_NAME)

In [6]:
vector_store = qdrant_index_manager.load_vector_index()

Loaded index 'uu-all-512-token'.


In [10]:
if vector_store:
    results = vector_store.similarity_search("Apa itu capital Papua?", k=5)
    for result in results:
        print(result.page_content, result.metadata)

DOC NAME: Undang-undang Nomor 19 Tahun 2007 Tentang Pembentukan Kabupaten Mamberamo Raya di Provinsi Papua | UNDANG-UNDANG

- 5 -
Pasal 7
 Burmeso sebagai Ibu Kota Kabupaten Mamberamo Raya berada
di Distrik Mamberamo Tengah.
Pasal 8
Cukup jelas.
Pasal 9
Peresmian kabupaten dan pelantikan Penjabat Bupati dapat
dilakukan 
secara 
bersamaan 
dan 
pelaksanaannya 
dapat
bertempat di ibu kota negara, atau ibu kota provinsi, atau ibu
kota kabupaten.
Pasal 10
Cukup jelas.
Pasal 11
Ayat (1)
 
Cukup jelas.
Ayat (2)
Penjabat 
Bupati 
Mamberamo 
Raya 
diusulkan 
oleh
Gubernur Papua dengan pertimbangan Bupati Sarmi dan
Bupati Waropen.
Ayat (3)
 
Cukup jelas.
Ayat (4)
 
Cukup jelas.
Ayat (5)
 
Cukup jelas.
Ayat (6)
Cukup jelas.
Pasal 12 . . .

Source: [Undang-undang Nomor 19 Tahun 2007 Tentang Pembentukan Kabupaten Mamberamo Raya di Provinsi Papua](https://peraturan.go.id/files/uu19-2007.pdf#page=23) {'title': 'Undang-undang Nomor 19 Tahun 2007 Tentang Pembentukan Kabupaten Mamberamo Raya di Provins

In [8]:
from utils.utils import set_gpu
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore", message=r"Some weights of .* were not initialized from the model checkpoint at .* and are newly initialized")
set_gpu(2)

Set GPU to 2


In [2]:
import numpy as np
from utils.database.qdrant import QdrantIndexManager
from utils.llm import get_embedding_model
from utils.utils import load_from_pickle
import pandas as pd

# === Load Test Set ===
testset = load_from_pickle('validated_testset')

# === Filter Rows with Metadata and Validate ===
# Function to check if metadata is valid
def is_metadata_valid(row):
    metadata = row['metadata']
    if not metadata or metadata[0].get('id') == '' or metadata[0].get('page_number') == '':
        return False
    return True

# Filter rows with valid metadata
testset['has_valid_metadata'] = testset.apply(is_metadata_valid, axis=1)
valid_testset = testset[(testset['has_valid_metadata']) & (testset['validate'] == 'A')]

# Generate Unique IDs
def generate_unique_id(row):
    metadata = row['metadata'][0]
    doc_id = metadata['id']
    page_number = metadata['page_number']
    return f"{doc_id}_page_{page_number}"

# Apply unique ID generation
valid_testset['unique_id'] = valid_testset.apply(generate_unique_id, axis=1)

# Prepare ground truth as a dictionary
ground_truth = {
    row['cleaned_question']: [row['unique_id']]
    for _, row in valid_testset.iterrows()
}

print(f"Number of valid rows with proper metadata and validate=A: {len(valid_testset)}")

Number of valid rows with proper metadata and validate=A: 2596


/tmp/ipykernel_3925242/1521620803.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_testset['unique_id'] = valid_testset.apply(generate_unique_id, axis=1)


In [2]:
# === Setup Configuration ===
CHUNK_SIZE = 512
CHUNK_OVERLAP = 128
EMBEDDING_MODEL_NAME = "BAAI/bge-m3"
COLLECTION_NAME_TEMPLATE = f"{CHUNK_SIZE}_{CHUNK_OVERLAP}_bge-m3"

RETRIEVAL_METHODS = ["Cosine Sim", "Euclidean Distance", "Manhattan Distance"]
K_VALUES = [1, 2, 5, 10]
RERANK_OPTIONS = [True, False]

In [3]:
# === Setup Qdrant ===
embed_model = get_embedding_model(model="BAAI/bge-m3")
COLLECTION_NAME = "uu-all-512-token"
qdrant_index_manager = QdrantIndexManager(embed_model, index_name=COLLECTION_NAME)
vector_store = qdrant_index_manager.load_vector_index()

Loaded index 'uu-all-512-token'.


In [4]:
# === Define Metrics Functions ===
def calculate_metrics(retrieved_ids, ground_truth_ids, k):
    retrieved_top_k = retrieved_ids[:k]
    relevant_ids = set(ground_truth_ids)
    
    # Precision@K
    precision = len(set(retrieved_top_k) & relevant_ids) / k if k > 0 else 0
    
    # Recall@K
    recall = len(set(retrieved_top_k) & relevant_ids) / len(relevant_ids) if relevant_ids else 0
    
    # MRR (Reciprocal Rank)
    reciprocal_rank = 0
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            reciprocal_rank = 1 / rank
            break
    
    # DCG@K
    dcg = sum(1 / np.log2(idx + 2) if doc in relevant_ids else 0 
              for idx, doc in enumerate(retrieved_top_k))
    ideal_dcg = sum(1 / np.log2(idx + 2) for idx in range(min(len(relevant_ids), k)))
    ndcg = dcg / ideal_dcg if ideal_dcg > 0 else 0

    return precision, recall, reciprocal_rank, ndcg

In [5]:
def calculate_map(retrieved_ids, ground_truth_ids):
    """
    Calculate Mean Average Precision (MAP) for a single query.
    :param retrieved_ids: List of retrieved document IDs.
    :param ground_truth_ids: Set of ground truth document IDs.
    :return: Average Precision (AP) for the query.
    """
    relevant_ids = set(ground_truth_ids)
    if not relevant_ids:
        return 0  # No relevant documents, AP is 0.

    precision_at_k = []
    num_relevant = 0

    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            num_relevant += 1
            precision_at_k.append(num_relevant / rank)

    return np.mean(precision_at_k) if precision_at_k else 0

In [6]:
from tqdm import tqdm 
# === Main Iteration ===
results = []

# Progress bar untuk seluruh kombinasi Retrieval Method dan K
total_iterations = len(RETRIEVAL_METHODS) * len(K_VALUES) * len(valid_testset)
with tqdm(total=total_iterations, desc="Processing Retrieval", unit="query") as pbar:
    for rm in RETRIEVAL_METHODS:
        for k in K_VALUES:
            # Define model name
            model_name = f"{CHUNK_SIZE}_{CHUNK_OVERLAP}_{EMBEDDING_MODEL_NAME.split('/')[-1]}_{rm}_K{k}"
            metrics_summary = {'Precision@K': [], 'Recall@K': [], 'MRR': [], 'nDCG@K': [], 'MAP': []}
            
            # Perform retrieval and calculate metrics
            for _, row in valid_testset.iterrows():
                question = row['cleaned_question']
                ground_truth_ids = ground_truth[question]

                # Retrieve results
                retrieval_results = vector_store.similarity_search(question, k=k)  # Customize for RM if needed
                retrieved_ids = [
                    f"{result.metadata['id']}_page_{result.metadata['page_number']}"
                    for result in retrieval_results
                ]

                # Calculate metrics
                precision, recall, mrr, ndcg = calculate_metrics(retrieved_ids, ground_truth_ids, k)
                average_precision = calculate_map(retrieved_ids, ground_truth_ids)
                
                metrics_summary['Precision@K'].append(precision)
                metrics_summary['Recall@K'].append(recall)
                metrics_summary['MRR'].append(mrr)
                metrics_summary['nDCG@K'].append(ndcg)
                metrics_summary['MAP'].append(average_precision)
                
                # Update progress bar
                pbar.update(1)
            
            # Summarize metrics
            average_metrics = {metric: np.mean(scores) for metric, scores in metrics_summary.items()}
            
            # Append to results
            results.append({
                'Model': model_name,
                **average_metrics
            })

Processing Retrieval: 100%|████████████████████████████████████████████████████| 31152/31152 [13:17<00:00, 39.07query/s]


In [7]:
# === Output Results ===
import pandas as pd

df_results = pd.DataFrame(results)
df_results.set_index('Model', inplace=True)
print("\n=== Evaluation Summary ===")
print(df_results)


=== Evaluation Summary ===
                                       Precision@K  Recall@K       MRR  \
Model                                                                    
512_128_bge-m3_Cosine Sim_K1              0.340139  0.340139  0.340139   
512_128_bge-m3_Cosine Sim_K2              0.229584  0.459168  0.399653   
512_128_bge-m3_Cosine Sim_K5              0.122342  0.611710  0.442206   
512_128_bge-m3_Cosine Sim_K10             0.070994  0.709938  0.455545   
512_128_bge-m3_Euclidean Distance_K1      0.340139  0.340139  0.340139   
512_128_bge-m3_Euclidean Distance_K2      0.229584  0.459168  0.399653   
512_128_bge-m3_Euclidean Distance_K5      0.122342  0.611710  0.442206   
512_128_bge-m3_Euclidean Distance_K10     0.070994  0.709938  0.455545   
512_128_bge-m3_Manhattan Distance_K1      0.340139  0.340139  0.340139   
512_128_bge-m3_Manhattan Distance_K2      0.229584  0.459168  0.399653   
512_128_bge-m3_Manhattan Distance_K5      0.122342  0.611710  0.442206   
512_128_bg

In [ ]:
# Save Results
results_df = pd.DataFrame(metrics_summary)
results_df.to_csv("retrieval_evaluation_results_with_map.csv", index=False)
print("Evaluation results saved to retrieval_evaluation_results_with_map.csv")

In [ ]:
# Real

In [1]:
from utils.utils import set_gpu
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore", message=r"Some weights of .* were not initialized from the model checkpoint at .* and are newly initialized")
set_gpu(2)

Set GPU to 2


In [2]:
import numpy as np
from utils.database.qdrant import QdrantIndexManager
from utils.llm import get_embedding_model
from utils.utils import load_from_pickle
import pandas as pd

# === Load Test Set ===
testset = load_from_pickle('validated_testset')

# === Filter Rows with Metadata and Validate ===
# Function to check if metadata is valid
def is_metadata_valid(row):
    metadata = row['metadata']
    if not metadata or metadata[0].get('id') == '' or metadata[0].get('page_number') == '':
        return False
    return True

# Filter rows with valid metadata
testset['has_valid_metadata'] = testset.apply(is_metadata_valid, axis=1)
valid_testset = testset[(testset['has_valid_metadata']) & (testset['validate'] == 'A')]

# Generate Unique IDs
def generate_unique_id(row):
    metadata = row['metadata'][0]
    doc_id = metadata['id']
    page_number = metadata['page_number']
    return f"{doc_id}_page_{page_number}"

# Apply unique ID generation
valid_testset['unique_id'] = valid_testset.apply(generate_unique_id, axis=1)

# Prepare ground truth as a dictionary
ground_truth = {
    row['cleaned_question']: [row['unique_id']]
    for _, row in valid_testset.iterrows()
}

print(f"Number of valid rows with proper metadata and validate=A: {len(valid_testset)}")

Number of valid rows with proper metadata and validate=A: 2596


/tmp/ipykernel_193528/1521620803.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_testset['unique_id'] = valid_testset.apply(generate_unique_id, axis=1)


In [3]:
# === Setup Embedding Model and Qdrant Index ===
embed_model = get_embedding_model(model="BAAI/bge-m3")
COLLECTION_NAME = "uu-all-512-128-bge"
qdrant_index_manager = QdrantIndexManager(embed_model, index_name=COLLECTION_NAME)
vector_store = qdrant_index_manager.load_vector_index()

# === Define Metrics Functions ===
def calculate_metrics(retrieved_ids, ground_truth_ids, k):
    retrieved_top_k = retrieved_ids[:k]
    relevant_ids = set(ground_truth_ids)
    
    # Precision@K
    precision = len(set(retrieved_top_k) & relevant_ids) / k if k > 0 else 0
    
    # Recall@K
    recall = len(set(retrieved_top_k) & relevant_ids) / len(relevant_ids) if relevant_ids else 0
    
    # MRR (Reciprocal Rank)
    reciprocal_rank = 0
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            reciprocal_rank = 1 / rank
            break
    
    # DCG@K
    dcg = sum(1 / np.log2(idx + 2) if doc in relevant_ids else 0 
              for idx, doc in enumerate(retrieved_top_k))
    ideal_dcg = sum(1 / np.log2(idx + 2) for idx in range(min(len(relevant_ids), k)))
    ndcg = dcg / ideal_dcg if ideal_dcg > 0 else 0

    return precision, recall, reciprocal_rank, ndcg

def calculate_map(retrieved_ids, ground_truth_ids):
    relevant_ids = set(ground_truth_ids)
    if not relevant_ids:
        return 0  # No relevant documents, AP is 0.

    precision_at_k = []
    num_relevant = 0

    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            num_relevant += 1
            precision_at_k.append(num_relevant / rank)

    return np.mean(precision_at_k) if precision_at_k else 0

Loaded index 'uu-all-512-128-bge'.


In [4]:
# === Main Iteration ===
K_VALUES = [1, 2, 5, 10]
LAMBDAS = [0.3, 0.5, 0.7]
metrics_summary = []

if vector_store:
    print("Starting evaluation process...")
    total_iterations = len(K_VALUES) * (len(LAMBDAS) + 1) * len(valid_testset)  # +1 for similarity search
    with tqdm(total=total_iterations, desc="Evaluating") as pbar:
        for k in K_VALUES:
            for lambda_mult in LAMBDAS:
                method_summary = {'K': k, 'Lambda': lambda_mult, 'Method': 'MMR', 'Metrics': {}}
                method_metrics = {'Precision@K': [], 'Recall@K': [], 'MRR': [], 'nDCG@K': [], 'MAP': []}

                for _, row in valid_testset.iterrows():
                    question = row['cleaned_question']
                    ground_truth_ids = ground_truth[question]

                    # Perform MMR Search
                    retrieval_results = vector_store.max_marginal_relevance_search(
                        question, k=k, lambda_mult=lambda_mult
                    )
                    retrieved_ids = [
                        f"{result.metadata['id']}_page_{result.metadata['page_number']}"
                        for result in retrieval_results
                    ]

                    # Calculate Metrics
                    precision, recall, mrr, ndcg = calculate_metrics(retrieved_ids, ground_truth_ids, k)
                    average_precision = calculate_map(retrieved_ids, ground_truth_ids)

                    method_metrics['Precision@K'].append(precision)
                    method_metrics['Recall@K'].append(recall)
                    method_metrics['MRR'].append(mrr)
                    method_metrics['nDCG@K'].append(ndcg)
                    method_metrics['MAP'].append(average_precision)

                    pbar.update(1)

                # Summarize metrics for this method
                method_summary['Metrics'] = {metric: np.mean(scores) for metric, scores in method_metrics.items()}
                metrics_summary.append(method_summary)

            # Similarity Search (for comparison)
            method_summary = {'K': k, 'Lambda': None, 'Method': 'Similarity', 'Metrics': {}}
            method_metrics = {'Precision@K': [], 'Recall@K': [], 'MRR': [], 'nDCG@K': [], 'MAP': []}

            for _, row in valid_testset.iterrows():
                question = row['cleaned_question']
                ground_truth_ids = ground_truth[question]

                # Perform Similarity Search
                retrieval_results = vector_store.similarity_search(question, k=k)
                retrieved_ids = [
                    f"{result.metadata['id']}_page_{result.metadata['page_number']}"
                    for result in retrieval_results
                ]

                # Calculate Metrics
                precision, recall, mrr, ndcg = calculate_metrics(retrieved_ids, ground_truth_ids, k)
                average_precision = calculate_map(retrieved_ids, ground_truth_ids)

                method_metrics['Precision@K'].append(precision)
                method_metrics['Recall@K'].append(recall)
                method_metrics['MRR'].append(mrr)
                method_metrics['nDCG@K'].append(ndcg)
                method_metrics['MAP'].append(average_precision)

                pbar.update(1)

            # Summarize metrics for this method
            method_summary['Metrics'] = {metric: np.mean(scores) for metric, scores in method_metrics.items()}
            metrics_summary.append(method_summary)

Starting evaluation process...


Evaluating: 100%|█████████████████████████████████████████████████████████████████| 41536/41536 [25:11<00:00, 27.47it/s]


In [7]:
# === Print Final Summary ===
print("\n=== Evaluation Summary ===")
for summary in metrics_summary:
    print(f"Method: {summary['Method']}, K: {summary['K']}, Lambda: {summary['Lambda']}")
    for metric, value in summary['Metrics'].items():
        print(f"  {metric}: {value:.4f}")
    print()


=== Evaluation Summary ===
Method: MMR, K: 1, Lambda: 0.3
  Precision@K: 0.3390
  Recall@K: 0.3390
  MRR: 0.3390
  nDCG@K: 0.3390
  MAP: 0.3390

Method: MMR, K: 1, Lambda: 0.5
  Precision@K: 0.3390
  Recall@K: 0.3390
  MRR: 0.3390
  nDCG@K: 0.3390
  MAP: 0.3390

Method: MMR, K: 1, Lambda: 0.7
  Precision@K: 0.3390
  Recall@K: 0.3390
  MRR: 0.3390
  nDCG@K: 0.3390
  MAP: 0.3390

Method: Similarity, K: 1, Lambda: None
  Precision@K: 0.3390
  Recall@K: 0.3390
  MRR: 0.3390
  nDCG@K: 0.3390
  MAP: 0.3390

Method: MMR, K: 2, Lambda: 0.3
  Precision@K: 0.1795
  Recall@K: 0.3590
  MRR: 0.3490
  nDCG@K: 0.3524
  MAP: 0.3490

Method: MMR, K: 2, Lambda: 0.5
  Precision@K: 0.1820
  Recall@K: 0.3640
  MRR: 0.3515
  nDCG@K: 0.3550
  MAP: 0.3515

Method: MMR, K: 2, Lambda: 0.7
  Precision@K: 0.2007
  Recall@K: 0.4014
  MRR: 0.3702
  nDCG@K: 0.3801
  MAP: 0.3702

Method: Similarity, K: 2, Lambda: None
  Precision@K: 0.2269
  Recall@K: 0.4538
  MRR: 0.3964
  nDCG@K: 0.4172
  MAP: 0.3964

Method: MMR,

In [1]:
import numpy as np
import pandas as pd
from utils.database.qdrant import QdrantIndexManager
from utils.llm import get_embedding_model
from utils.utils import load_from_pickle
from tqdm import tqdm

# === Load Test Set ===
testset = load_from_pickle('validated_testset')

# === Filter Rows with Metadata and Validate ===
def is_metadata_valid(row):
    metadata = row['metadata']
    if not metadata or metadata[0].get('id') == '' or metadata[0].get('page_number') == '':
        return False
    return True

testset['has_valid_metadata'] = testset.apply(is_metadata_valid, axis=1)
valid_testset = testset[(testset['has_valid_metadata']) & (testset['validate'] == 'A')]

def generate_unique_id(row):
    metadata = row['metadata'][0]
    doc_id = metadata['id']
    page_number = metadata['page_number']
    return f"{doc_id}_page_{page_number}"

valid_testset['unique_id'] = valid_testset.apply(generate_unique_id, axis=1)

# Prepare ground truth
ground_truth = {
    row['cleaned_question']: [row['unique_id']]
    for _, row in valid_testset.iterrows()
}

print(f"Number of valid rows with proper metadata and validate=A: {len(valid_testset)}")

Number of valid rows with proper metadata and validate=A: 2596


/tmp/ipykernel_327473/766672603.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_testset['unique_id'] = valid_testset.apply(generate_unique_id, axis=1)


In [2]:
# === Setup Embedding Model and Qdrant Index ===
embed_model = get_embedding_model(model="BAAI/bge-m3")
COLLECTION_NAME = "uu-all-512-128-bge"
qdrant_index_manager = QdrantIndexManager(embed_model, index_name=COLLECTION_NAME)
vector_store = qdrant_index_manager.load_vector_index()

# === Define Metrics Functions ===
def calculate_metrics(retrieved_ids, ground_truth_ids, k):
    retrieved_top_k = retrieved_ids[:k]
    relevant_ids = set(ground_truth_ids)
    
    # Precision@K
    precision = len(set(retrieved_top_k) & relevant_ids) / k if k > 0 else 0
    
    # Recall@K
    recall = len(set(retrieved_top_k) & relevant_ids) / len(relevant_ids) if relevant_ids else 0
    
    # MRR (Reciprocal Rank)
    reciprocal_rank = 0
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            reciprocal_rank = 1 / rank
            break
    
    # DCG@K
    dcg = sum(1 / np.log2(idx + 2) if doc in relevant_ids else 0 
              for idx, doc in enumerate(retrieved_top_k))
    ideal_dcg = sum(1 / np.log2(idx + 2) for idx in range(min(len(relevant_ids), k)))
    ndcg = dcg / ideal_dcg if ideal_dcg > 0 else 0

    return precision, recall, reciprocal_rank, ndcg

def calculate_map(retrieved_ids, ground_truth_ids):
    relevant_ids = set(ground_truth_ids)
    if not relevant_ids:
        return 0  # No relevant documents, AP is 0.
    precision_at_k = []
    num_relevant = 0
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            num_relevant += 1
            precision_at_k.append(num_relevant / rank)
    return np.mean(precision_at_k) if precision_at_k else 0

Loaded index 'uu-all-512-128-bge'.


In [3]:
# === Main Iteration ===
K_VALUES = [1, 2, 5, 10]
LAMBDAS = [0.3, 0.5, 0.7]
metrics_summary = []

if vector_store:
    print("Starting evaluation process...")
    total_iterations = len(K_VALUES) * (len(LAMBDAS) + 1) * len(valid_testset)  # +1 for similarity search per K
    with tqdm(total=total_iterations, desc="Evaluating") as pbar:
        for k in K_VALUES:
            # MMR retrieval
            for lambda_mult in LAMBDAS:
                method_summary = {'K': k, 'Lambda': lambda_mult, 'Method': 'MMR', 'Metrics': {}}
                method_metrics = {'Precision@K': [], 'Recall@K': [], 'MRR': [], 'nDCG@K': [], 'MAP': []}

                for _, row in valid_testset.iterrows():
                    question = row['cleaned_question']
                    ground_truth_ids = ground_truth[question]

                    retrieval_results = vector_store.max_marginal_relevance_search(
                        question, k=k, lambda_mult=lambda_mult
                    )
                    retrieved_ids = [
                        f"{result.metadata['id']}_page_{result.metadata['page_number']}"
                        for result in retrieval_results
                    ]

                    precision, recall, mrr, ndcg = calculate_metrics(retrieved_ids, ground_truth_ids, k)
                    average_precision = calculate_map(retrieved_ids, ground_truth_ids)

                    method_metrics['Precision@K'].append(precision)
                    method_metrics['Recall@K'].append(recall)
                    method_metrics['MRR'].append(mrr)
                    method_metrics['nDCG@K'].append(ndcg)
                    method_metrics['MAP'].append(average_precision)
                    pbar.update(1)

                method_summary['Metrics'] = {metric: np.mean(scores) for metric, scores in method_metrics.items()}
                metrics_summary.append(method_summary)

            # Similarity Search (Cosine)
            method_summary = {'K': k, 'Lambda': None, 'Method': 'Similarity', 'Metrics': {}}
            method_metrics = {'Precision@K': [], 'Recall@K': [], 'MRR': [], 'nDCG@K': [], 'MAP': []}

            for _, row in valid_testset.iterrows():
                question = row['cleaned_question']
                ground_truth_ids = ground_truth[question]

                retrieval_results = vector_store.similarity_search(question, k=k)
                retrieved_ids = [
                    f"{result.metadata['id']}_page_{result.metadata['page_number']}"
                    for result in retrieval_results
                ]

                precision, recall, mrr, ndcg = calculate_metrics(retrieved_ids, ground_truth_ids, k)
                average_precision = calculate_map(retrieved_ids, ground_truth_ids)

                method_metrics['Precision@K'].append(precision)
                method_metrics['Recall@K'].append(recall)
                method_metrics['MRR'].append(mrr)
                method_metrics['nDCG@K'].append(ndcg)
                method_metrics['MAP'].append(average_precision)
                pbar.update(1)

            method_summary['Metrics'] = {metric: np.mean(scores) for metric, scores in method_metrics.items()}
            metrics_summary.append(method_summary)

Starting evaluation process...


Evaluating: 100%|█████████████████████████████████████████████████████████████████| 41536/41536 [25:08<00:00, 27.54it/s]


In [4]:
# === Constructing the Final DataFrame ===
# Naming convention: CS_CO_EM_RM_K_RR
# From COLLECTION_NAME = "uu-all-512-128-bge" we deduce:
CS_CO = "512_128"
EM = "bge"
RR = "no"  # Currently not using re-ranking

df_results = pd.DataFrame()

for summary in metrics_summary:
    k = summary['K']
    method = summary['Method']
    lam = summary['Lambda']
    if method == "Similarity":
        RM = "cos"
    else:
        RM = f"mmr{lam}"

    # Construct experiment code
    # Example: "512_128_bge_mmr0.3_1_no"
    experiment_code = f"{CS_CO}_{EM}_{RM}_{k}_{RR}"

    # Insert the metrics into the DataFrame
    for metric, val in summary['Metrics'].items():
        df_results.loc[metric, experiment_code] = val

# Display the final results DataFrame
df_results

df_results.to_csv("evaluation_results_512_128_no.csv")

,512_128_bge_mmr0.3_1_no,512_128_bge_mmr0.5_1_no,512_128_bge_mmr0.7_1_no,512_128_bge_cos_1_no,512_128_bge_mmr0.3_2_no,512_128_bge_mmr0.5_2_no,512_128_bge_mmr0.7_2_no,512_128_bge_cos_2_no,512_128_bge_mmr0.3_5_no,512_128_bge_mmr0.5_5_no,512_128_bge_mmr0.7_5_no,512_128_bge_cos_5_no,512_128_bge_mmr0.3_10_no,512_128_bge_mmr0.5_10_no,512_128_bge_mmr0.7_10_no,512_128_bge_cos_10_no
Precision@K,0.338983,0.338983,0.338983,0.338983,0.179507,0.182011,0.200693,0.226888,0.084515,0.090524,0.104545,0.122804,0.054122,0.059515,0.065216,0.070300
Recall@K,0.338983,0.338983,0.338983,0.338983,0.359014,0.364022,0.401387,0.453775,0.422573,0.452619,0.522727,0.614022,0.541217,0.595146,0.652157,0.703005
MRR,0.338983,0.338983,0.338983,0.338983,0.348998,0.351502,0.370185,0.396379,0.365806,0.375167,0.402953,0.440973,0.381144,0.393307,0.420157,0.452978
nDCG@K,0.338983,0.338983,0.338983,0.338983,0.352350,0.355024,0.380057,0.417242,0.383863,0.399610,0.443459,0.503549,0.430911,0.455317,0.496204,0.542806
MAP,0.338983,0.338983,0.338983,0.338983,0.348998,0.351502,0.370185,0.396379,0.365196,0.373966,0.400746,0.438626,0.374097,0.385548,0.411697,0.443745


In [1]:
import numpy as np
import pandas as pd
from utils.database.qdrant import QdrantIndexManager
from utils.llm import get_embedding_model
from utils.utils import load_from_pickle
from tqdm import tqdm

from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders.huggingface import HuggingFaceCrossEncoder

# === Load Test Set ===
testset = load_from_pickle('validated_testset')

# === Filter Rows with Metadata and Validate ===
def is_metadata_valid(row):
    metadata = row['metadata']
    if not metadata or metadata[0].get('id') == '' or metadata[0].get('page_number') == '':
        return False
    return True

testset['has_valid_metadata'] = testset.apply(is_metadata_valid, axis=1)
valid_testset = testset[(testset['has_valid_metadata']) & (testset['validate'] == 'A')]

def generate_unique_id(row):
    metadata = row['metadata'][0]
    doc_id = metadata['id']
    page_number = metadata['page_number']
    return f"{doc_id}_page_{page_number}"

valid_testset['unique_id'] = valid_testset.apply(generate_unique_id, axis=1)

# Prepare ground truth
ground_truth = {
    row['cleaned_question']: [row['unique_id']]
    for _, row in valid_testset.iterrows()
}

print(f"Number of valid rows with proper metadata and validate=A: {len(valid_testset)}")

Number of valid rows with proper metadata and validate=A: 2596


/tmp/ipykernel_604254/137421604.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_testset['unique_id'] = valid_testset.apply(generate_unique_id, axis=1)


In [2]:
# === Setup Embedding Model and Qdrant Index ===
embed_model = get_embedding_model(model="BAAI/bge-m3")
COLLECTION_NAME = "uu-all-512-128-bge"
qdrant_index_manager = QdrantIndexManager(embed_model, index_name=COLLECTION_NAME)
vector_store = qdrant_index_manager.load_vector_index()

# === Define Metrics Functions ===
def calculate_metrics(retrieved_ids, ground_truth_ids, k):
    retrieved_top_k = retrieved_ids[:k]
    relevant_ids = set(ground_truth_ids)
    
    # Precision@K
    precision = len(set(retrieved_top_k) & relevant_ids) / k if k > 0 else 0
    
    # Recall@K
    recall = len(set(retrieved_top_k) & relevant_ids) / len(relevant_ids) if relevant_ids else 0
    
    # MRR (Reciprocal Rank)
    reciprocal_rank = 0
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            reciprocal_rank = 1 / rank
            break
    
    # DCG@K
    dcg = sum(1 / np.log2(idx + 2) if doc in relevant_ids else 0 
              for idx, doc in enumerate(retrieved_top_k))
    ideal_dcg = sum(1 / np.log2(idx + 2) for idx in range(min(len(relevant_ids), k)))
    ndcg = dcg / ideal_dcg if ideal_dcg > 0 else 0

    return precision, recall, reciprocal_rank, ndcg

def calculate_map(retrieved_ids, ground_truth_ids):
    relevant_ids = set(ground_truth_ids)
    if not relevant_ids:
        return 0  # No relevant documents, AP is 0.
    precision_at_k = []
    num_relevant = 0
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            num_relevant += 1
            precision_at_k.append(num_relevant / rank)
    return np.mean(precision_at_k) if precision_at_k else 0

Loaded index 'uu-all-512-128-bge'.


In [ ]:
# === Main Iteration ===
K_VALUES = [1, 2, 5, 10]
LAMBDAS = [0.3, 0.5, 0.7]
metrics_summary = []

if vector_store:
    print("Starting evaluation process...")
    total_iterations = len(K_VALUES) * (len(LAMBDAS) + 1) * len(valid_testset)  # +1 for similarity search per K
    
    # Initialize the cross-encoder model once
    model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
    
    with tqdm(total=total_iterations, desc="Evaluating") as pbar:
        for k in K_VALUES:
            # Initialize the reranker for this K
            cross_encoder_compressor = CrossEncoderReranker(model=model, top_n=k)
            
            # MMR retrieval + re-ranking
            for lambda_mult in LAMBDAS:
                method_summary = {'K': k, 'Lambda': lambda_mult, 'Method': 'MMR', 'Metrics': {}}
                method_metrics = {'Precision@K': [], 'Recall@K': [], 'MRR': [], 'nDCG@K': [], 'MAP': []}

                for _, row in valid_testset.iterrows():
                    question = row['cleaned_question']
                    ground_truth_ids = ground_truth[question]

                    retrieval_results = vector_store.max_marginal_relevance_search(
                        question, k=k, lambda_mult=lambda_mult
                    )

                    # Apply reranker
                    reranked_docs = cross_encoder_compressor.compress_documents(documents=retrieval_results, query=question)
                    
                    retrieved_ids = [
                        f"{doc.metadata['id']}_page_{doc.metadata['page_number']}"
                        for doc in reranked_docs
                    ]

                    precision, recall, mrr, ndcg = calculate_metrics(retrieved_ids, ground_truth_ids, k)
                    average_precision = calculate_map(retrieved_ids, ground_truth_ids)

                    method_metrics['Precision@K'].append(precision)
                    method_metrics['Recall@K'].append(recall)
                    method_metrics['MRR'].append(mrr)
                    method_metrics['nDCG@K'].append(ndcg)
                    method_metrics['MAP'].append(average_precision)
                    pbar.update(1)

                method_summary['Metrics'] = {metric: np.mean(scores) for metric, scores in method_metrics.items()}
                metrics_summary.append(method_summary)

            # Similarity Search (Cosine) + re-ranking
            method_summary = {'K': k, 'Lambda': None, 'Method': 'Similarity', 'Metrics': {}}
            method_metrics = {'Precision@K': [], 'Recall@K': [], 'MRR': [], 'nDCG@K': [], 'MAP': []}

            for _, row in valid_testset.iterrows():
                question = row['cleaned_question']
                ground_truth_ids = ground_truth[question]

                retrieval_results = vector_store.similarity_search(question, k=40)  # fetch more to rerank top k
                # Rerank to top k
                reranked_docs = cross_encoder_compressor.compress_documents(documents=retrieval_results, query=question)

                retrieved_ids = [
                    f"{doc.metadata['id']}_page_{doc.metadata['page_number']}"
                    for doc in reranked_docs
                ]

                precision, recall, mrr, ndcg = calculate_metrics(retrieved_ids, ground_truth_ids, k)
                average_precision = calculate_map(retrieved_ids, ground_truth_ids)

                method_metrics['Precision@K'].append(precision)
                method_metrics['Recall@K'].append(recall)
                method_metrics['MRR'].append(mrr)
                method_metrics['nDCG@K'].append(ndcg)
                method_metrics['MAP'].append(average_precision)
                pbar.update(1)

            method_summary['Metrics'] = {metric: np.mean(scores) for metric, scores in method_metrics.items()}
            metrics_summary.append(method_summary)

# === Constructing the Final DataFrame ===
# Naming convention: CS_CO_EM_RM_K_RR
# From COLLECTION_NAME = "uu-all-512-128-bge" we deduce:
CS_CO = "512_128"
EM = "bge"
RR = "yes"  # Now we have re-ranking by default

df_results = pd.DataFrame()

for summary in metrics_summary:
    k = summary['K']
    method = summary['Method']
    lam = summary['Lambda']
    if method == "Similarity":
        RM = "cos"
    else:
        RM = f"mmr{lam}"

    # Construct experiment code with RR = yes
    experiment_code = f"{CS_CO}_{EM}_{RM}_{k}_{RR}"

    # Insert the metrics into the DataFrame
    for metric, val in summary['Metrics'].items():
        df_results.loc[metric, experiment_code] = val

# Display and save the final results DataFrame
df_results

df_results.to_csv("evaluation_results_256_0_no.csv")

Starting evaluation process...


Evaluating:  20%|████████████▊                                                   | 8282/41536 [14:08<6:31:46,  1.41it/s]